<a href="https://colab.research.google.com/github/adrian-lo-te/Sprints/blob/main/S7_N1_E2_corregido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

El que he fet en aquesta funció és definir un diccionari de diccionaris amb els comportaments, segments, missatges i factors corresponents. Després, l'he convertit en data frame mitjançant .from_dict() i l'he unit amb la taula de clients utilitzant pd.merge() a partir de la columna en comú "codi_comportament".

In [ ]:
def obtenir_segment_i_factor(llista_clients):
  df_clients = pd.DataFrame(llista_clients)

  comportaments_dict = {
      0: {"segment": "Navegant actiu", "missatge": "Descobreix novetats que et poden interessar!", "factor_codi": 0.10},
      1: {"segment": "Interés inicial", "missatge": "Encara hi ets a temps! Completa la teva compra.", "factor_codi": 0.25},
      2: {"segment": "Intenció alta", "missatge": "Necessites ajuda amb el pagament? Finalitza ara.", "factor_codi": 0.50},
      3: {"segment": "Cistella abandonada recent", "missatge": "T'hem guardat la cistella! Aconsegueix un 5% de descompte.", "factor_codi": 0.75},
      4: {"segment": "Cistella abandonada freda", "missatge": "Última oportunitat! No perdis els teus productes especials.", "factor_codi": 0.90}
  }

  df_comportaments = pd.DataFrame.from_dict(comportaments_dict, orient="index").reset_index(names="codi_comportament")
  df_merged = pd.merge(df_clients, df_comportaments, on="codi_comportament", how="left")

  return df_merged

Aquesta funció fa una operació similar a l'anterior: a partir d'un diccionari de diccionaris amb els grups d'edat i el seu factor associat, el transforma en un data frame amb .from_dict() i el conjunta amb la taula principal mitjançant pd.merge() a la columna "grup_edat".

In [ ]:
def obtenir_factor_edat(df_clients):
  edats_dict = {
      "18-25": {"factor_edat": 1.20},
      "26-40": {"factor_edat": 1.00},
      "41-60": {"factor_edat": 0.85},
      "60+": {"factor_edat": 0.70}
  }

  df_edats = pd.DataFrame.from_dict(edats_dict, orient="index").reset_index(names="grup_edat")
  df_merged = pd.merge(df_clients, df_edats, on="grup_edat", how="left")

  return df_merged

Aquí estic creant una nova columna on els seus valors son la multiplicació de dues altres columnes.

In [ ]:
def calcular_risc(df_clients):
  df_clients["factor_risc"] = df_clients["factor_edat"] * df_clients["factor_codi"]
  return df_clients

Aquesta funció crea un nou data frame a partir del que li donem orquestrant les funcions anteriors.

In [ ]:
def processar_clients(llista_clients):
  client_segment = obtenir_segment_i_factor(llista_clients)
  client_factor = obtenir_factor_edat(client_segment)
  client_total = calcular_risc(client_factor)

  return client_total

Aquí mostrem el data frame.

In [ ]:
def mostrar_dataframe(df_clients):
  display(df_clients)

Aquesta funció s'encarrega de processar i mostrar per pantalla un informe analític de màrqueting basat en les dades del DataFrame de clients. En primer lloc, ordena els clients de major a menor segons el seu risc d'abandonament i presenta els tres primers que s'han de contactar de manera prioritària, mostrant el seu nom, identificador, nivell de risc i el missatge personalitzat que se'ls enviarà. A continuació, analitza l'impacte de l'edat filtrant els usuaris amb un comportament específic per demostrar com els clients més joves tenen un factor multiplicador més alt que incrementa el seu risc final en comparació amb els perfils sènior. Després, identifica aquells usuaris amb un risc o interès molt baix per determinar a quins perfils no s'ha de destinar pressupost publicitari. Finalment, selecciona de manera directa el client que encabeix la llista per tenir el major risc i urgència, destacant-lo com el perfil més rendible de recuperar juntament amb una breu justificació basada en el seu comportament de compra i el seu grup d'edat.

In [ ]:
def mostrar_respostes(df_clients):

  # 1. Quins clients s'han de contactar primer?
  top_contactar = df_clients.sort_values(by="factor_risc", ascending=False)
  print("\n1. Clients a contactar primer (ordenats per major risc d'abandonament):")
  for _, row in top_contactar.head(3).iterrows():
      print(
          f"  {row['nom']} (ID: {row['id_client']}) | Risc: {row['factor_risc']} | Missatge: '{row['missatge']}'"
      )

  # 2. Diferències per edat amb el mateix comportament
  print(
        "\n2. Impacte de l'edat en el risc final (Exemple amb Codi 4 - Abandó >24h):"
    )
  codi_4 = df_clients[df_clients["codi_comportament"] == 4].sort_values(
        by="factor_risc", ascending=False
    )
  for _, row in codi_4.iterrows():
      print(
            f"   {row['nom']} ({row['grup_edat']} anys) -> Risc: {row['factor_risc']} (Factor edat: {row['factor_edat']})"
        )
  print(
        "   > Conclusió: Els usuaris més joves tenen un factor multiplicador més alt (1.20) que incrementa el seu risc final respecte als sènior (0.70)."
    )

  # 3. Clients amb poques possibilitats (Poc interès)
  menys_interes = df_clients[df_clients["factor_risc"] <= 0.25]
  print("\n3. Clients amb baix risc/interès (On NO invertir pressupost):")
  for _, row in menys_interes.iterrows():
      print(
            f"    {row['nom']} (ID: {row['id_client']}) | Codi {row['codi_comportament']} | Risc final: {row['factor_risc']}"
        )

  # 4. Client més rendible de recuperar
  mes_rendible = top_contactar.iloc[0]
  print(
      f"\n4. Client més rendible de recuperar: **{mes_rendible['nom']}** (ID: {mes_rendible['id_client']})"
  )
  print(
      f"   > Raó: Presenta el risc final més alt ({mes_rendible['factor_risc']}) al tenir la cistella abandonada fa més de 24h i pertànyer al segment jove (18-25) amb alta volatilitat i sensibilitat al retargeting."
  )

Finalment, a aquesta funció s'integra la resta de funcions per mostrar el resultat i les preguntes.

In [ ]:
def main_marketing():
    clients = [
        {
            "id_client": 1,
            "nom": "Laura Serra",
            "edat": 23,
            "grup_edat": "18-25",
            "codi_comportament": 4,
        },
        {
            "id_client": 2,
            "nom": "Marc Vidal",
            "edat": 35,
            "grup_edat": "26-40",
            "codi_comportament": 3,
        },
        {
            "id_client": 3,
            "nom": "Ana López",
            "edat": 29,
            "grup_edat": "26-40",
            "codi_comportament": 2,
        },
        {
            "id_client": 4,
            "nom": "Joan Riera",
            "edat": 19,
            "grup_edat": "18-25",
            "codi_comportament": 2,
        },
        {
            "id_client": 5,
            "nom": "Mónica Pérez",
            "edat": 47,
            "grup_edat": "41-60",
            "codi_comportament": 4,
        },
        {
            "id_client": 6,
            "nom": "Luis García",
            "edat": 52,
            "grup_edat": "41-60",
            "codi_comportament": 1,
        },
        {
            "id_client": 7,
            "nom": "Pilar Sánchez",
            "edat": 61,
            "grup_edat": "60+",
            "codi_comportament": 3,
        },
        {
            "id_client": 8,
            "nom": "Eva Martín",
            "edat": 38,
            "grup_edat": "26-40",
            "codi_comportament": 0,
        },
        {
            "id_client": 9,
            "nom": "Diego Romero",
            "edat": 24,
            "grup_edat": "18-25",
            "codi_comportament": 1,
        },
        {
            "id_client": 10,
            "nom": "Núria Costa",
            "edat": 33,
            "grup_edat": "26-40",
            "codi_comportament": 4,
        },
    ]

    df_resultat = processar_clients(clients)
    mostrar_dataframe(df_resultat)
    mostrar_respostes(df_resultat)

In [ ]:
main_marketing()

,id_client,nom,edat,grup_edat,codi_comportament,segment,missatge,factor_codi,factor_edat,factor_risc
0,1,Laura Serra,23,18-25,4,Cistella abandonada freda,Última oportunitat! No perdis els teus product...,0.90,1.20,1.0800
1,2,Marc Vidal,35,26-40,3,Cistella abandonada recent,T'hem guardat la cistella! Aconsegueix un 5% d...,0.75,1.00,0.7500
2,3,Ana López,29,26-40,2,Intenció alta,Necessites ajuda amb el pagament? Finalitza ara.,0.50,1.00,0.5000
3,4,Joan Riera,19,18-25,2,Intenció alta,Necessites ajuda amb el pagament? Finalitza ara.,0.50,1.20,0.6000
4,5,Mónica Pérez,47,41-60,4,Cistella abandonada freda,Última oportunitat! No perdis els teus product...,0.90,0.85,0.7650
5,6,Luis García,52,41-60,1,Interés inicial,Encara hi ets a temps! Completa la teva compra.,0.25,0.85,0.2125
6,7,Pilar Sánchez,61,60+,3,Cistella abandonada recent,T'hem guardat la cistella! Aconsegueix un 5% d...,0.75,0.70,0.5250
7,8,Eva Martín,38,26-40,0,Navegant actiu,Descobreix novetats que et poden interessar!,0.10,1.00,0.1000
8,9,Diego Romero,24,18-25,1,Interés inicial,Encara hi ets a temps! Completa la teva compra.,0.25,1.20,0.3000
9,10,Núria Costa,33,26-40,4,Cistella abandonada freda,Última oportunitat! No perdis els teus product...,0.90,1.00,0.9000



1. Clients a contactar primer (ordenats per major risc d'abandonament):
  Laura Serra (ID: 1) | Risc: 1.08 | Missatge: 'Última oportunitat! No perdis els teus productes especials.'
  Núria Costa (ID: 10) | Risc: 0.9 | Missatge: 'Última oportunitat! No perdis els teus productes especials.'
  Mónica Pérez (ID: 5) | Risc: 0.765 | Missatge: 'Última oportunitat! No perdis els teus productes especials.'

2. Impacte de l'edat en el risc final (Exemple amb Codi 4 - Abandó >24h):
   Laura Serra (18-25 anys) -> Risc: 1.08 (Factor edat: 1.2)
   Núria Costa (26-40 anys) -> Risc: 0.9 (Factor edat: 1.0)
   Mónica Pérez (41-60 anys) -> Risc: 0.765 (Factor edat: 0.85)
   > Conclusió: Els usuaris més joves tenen un factor multiplicador més alt (1.20) que incrementa el seu risc final respecte als sènior (0.70).

3. Clients amb baix risc/interès (On NO invertir pressupost):
    Luis García (ID: 6) | Codi 1 | Risc final: 0.2125
    Eva Martín (ID: 8) | Codi 0 | Risc final: 0.1

4. Client més rendible de 